# TP6 — Regularización en CNNs para ADN

                Notebook-esqueleto didáctico armado a partir de `TP6-Regulariz.pdf`.

                **Idea de uso**
                - La consigna completa está integrada en el orden del PDF, agrupada por objetivos/ejercicios.
                - Cada bloque de consigna tiene abajo celdas **TODO** mínimas para que el equipo complete.
                - No hay resolución implementada: las celdas usan plantillas y `raise NotImplementedError`.
                - El TP6 reutiliza datos, OHE, modelo CNN y métricas del TP5; pueden copiar funciones terminadas del notebook del TP5 cuando las tengan.
                - Completen también las celdas de análisis: son parte central del TP.

## 0. Setup general

                Ejecuten esta celda al inicio. El TP requiere PyTorch y probablemente `scikit-learn` para métricas.

                Si falta alguna librería, actualicen el entorno antes de correr el notebook.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Opcional para métricas:
# from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
# from sklearn.metrics import roc_auc_score, average_precision_score
# from sklearn.metrics import roc_curve, precision_recall_curve

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "TP6-Regularizacion":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

BED_PATH = PROJECT_ROOT / "jun_np_chr22_GRCh38.bed"
POSITIVE_FASTA_PATH = PROJECT_ROOT / "sequences_upper.fasta"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("BED:", BED_PATH)
print("FASTA positivos:", POSITIVE_FASTA_PATH)


## Consigna — Trabajo Práctico 6 — Introducción

Trabajo Práctico 6
Regularización
Objetivo:  El  objetivo  del  presente  TP  es  familiarizarse  con  las  denominadas  técnicas  de
regularización  (batch  normalization,  L1,  L2,  Drop-out)  que  lo  que  buscan  es  evitar  el
“overfitting”.  También  analizaremos  como  determinar  el  overfitting  y  cómo  el  mismo  se
relaciona con la complejidad de la red.
Para hacer este TP utilizaremos los datos y modelo del TP anterior, o sea vamos a analizar y
evitar el overfitting en la CNN utilizada para determinar sitios de unión al FT. Recomendamos
trabajar con secuencias de no más de 100nt (para agilizar la obtención de los resultados). 
Para determinar el overfitting es clave que separemos nuestros datos en un conjunto que sea
para entrenar, y otro que sea sólo para testear/validar. Considerando que el Chr22 posee 561
secuencias positivas podemos seleccionar 61 al azar para el el testeo (a las que agregaremos
igual  número de  secuencias  negativas  generadas  como en  el  TP  anterior)  y 500  para el
entrenamiento.

## 1. Base común del TP6 — Reutilizar TP5

                Antes de estudiar regularización, reconstruyan la base del TP5:
                - lectura de datos,
                - generación de negativos,
                - OHE con largo fijo,
                - split train/test,
                - CNN base,
                - función de entrenamiento,
                - función de evaluación.

                Mantengan esta sección simple: el objetivo del TP6 es comparar overfitting y regularización, no volver a pelearse con el dataset.

In [ ]:
# TODO: copiar/adaptar desde TP5 las funciones ya resueltas.

def leer_fasta(path):
    raise NotImplementedError


def generar_negativos(sequences_pos, estrategia="tp5", rng=None):
    raise NotImplementedError


def one_hot_encode_sequence(seq, nt_len):
    raise NotImplementedError


def secuencias_a_tensor(sequences, nt_len):
    raise NotImplementedError


def crear_dataset_tp6(nt_len=100, n_test_pos=61, random_state=42):
    """Devuelve train, train_y, test, test_y y dataframes auxiliares para el TP6."""
    raise NotImplementedError


# TODO: construir dataset base.
nt_len = 100
# train, train_y, test, test_y, train_df, test_df = crear_dataset_tp6(nt_len=nt_len)


### TODO — CNN base y helpers de entrenamiento/evaluación

                Definan acá una CNN equivalente a la del TP5 y helpers reutilizables para todos los ejercicios.

                Sugerencia: separen bien:
                - `entrenar_modelo(...)`
                - `evaluar_modelo(...)`
                - `graficar_loss(...)`
                - `graficar_roc_pr(...)`

In [ ]:
class DNA_CNN_Base(nn.Module):
    def __init__(self, nt_len, n_filters=8, kernel_size=5):
        super().__init__()
        # TODO: definir conv1 y fc1.
        raise NotImplementedError

    def forward(self, x):
        # TODO: conv + ReLU + flatten + fc + sigmoid.
        return x


def entrenar_modelo(model, train, train_y, test, test_y, epochs=100, batch_size=10, lr=0.001, weight_decay=0.0):
    """Entrena un modelo y devuelve historial de train/test loss."""
    raise NotImplementedError


def evaluar_modelo(model, X, y, threshold=0.5):
    """Devuelve scores, clases predichas y métricas principales."""
    raise NotImplementedError


def graficar_loss(history, title="Curvas de pérdida"):
    raise NotImplementedError


def graficar_roc_pr(resultados, title_suffix=""):
    raise NotImplementedError


## Consigna — 1. ¿cómo evidenciar el overfitting?

1. ¿cómo evidenciar el overfitting?
Un primer análisis simple consiste en analizar como decae la LF en el train set, y como decae
en el test set (como en el siguiente  ejemplo). (si usted fue a las clases teóricas  debería
entender por que ambas curvas tienen esa forma).
Es claro que más allá de unas 25-30 épocas la red comienza a sobreajustar.
La confirmación del overfitting, nos la puede  dar el análisis  comparativo  de las diferentes
métricas de performance (confusión matrix, accuracy, tpr etc.)

Por ejemplo, a contnuación se muestran ambas curvas ROC
Con AUCs de 0.99 para el train y 0.85 para el test.
Habiendo identificado el problema, vamos a analizar e implementar diferentes medidas para
corregirlo.  Las  mismas  se  conocen,  cómo  usted  ya  debería  saber,  como  medidas  de
Regularización.
Objetivo  1.  Nuestro  primer  objetivo  será  analizar  el  overfitting  en  función  de  los
hiperparámetros de la CNN (la cantidad de kernels, y su dimensión).
Ejercicio 6.1 Utilizando los datos y el tipo de CNN vistos en el TP anterior construya una red y
analicé la performance de las diferentes métricas (LF, Accuracy, ROC-AUC, CF, PR-AUC etc.)
para determinar si hay (o no) overfitting. Luego Analicé el efecto de la cantidad de kernels
(utilicé valores en en el rango 2-24), y su dimensión (3 a 20nt). Si puede muestre los resultados
en una tabla/gráfico. 
¿que puede decir del overfitting en relación con la complejidad de la red?  
¿cuál es la mejor red que puede obtener para discriminar el set entrenamiento? 
¿Que pasa si ahora además busca minimizar el overfittng?
Anote su respuesta a estas preguntas para discutir en clase.

### TODO 6.1.a — Evidenciar overfitting con una CNN base

                Entrenen una CNN base y comparen train vs test:
                - curvas de LF,
                - matriz de confusión,
                - Accuracy,
                - ROC-AUC,
                - PR-AUC,
                - cualquier otra métrica útil.

                Busquen si aparece el patrón típico: train mejora mucho mientras test deja de mejorar o empeora.

In [ ]:
# TODO: elegir una arquitectura base para observar overfitting.
base_config = {
    "n_filters": None,
    "kernel_size": None,
    "epochs": None,
    "lr": None,
}

# TODO: entrenar modelo base.
# model_base = DNA_CNN_Base(...)
# model_base, history_base = entrenar_modelo(...)

# TODO: evaluar train y test.
# train_metrics_base = evaluar_modelo(model_base, train, train_y)
# test_metrics_base = evaluar_modelo(model_base, test, test_y)

# TODO: graficar loss, ROC, PR y matriz de confusión.

raise NotImplementedError("Completar evidencia inicial de overfitting.")


### TODO 6.1.b — Barrido de complejidad: filtros y kernel size

                Analicen el efecto de:
                - cantidad de kernels/filtros en el rango `2–24`,
                - dimensión de kernel en el rango `3–20 nt`.

                Guarden resultados en una tabla y/o gráficos. Incluyan métricas de train y test para poder medir brecha de overfitting.

In [ ]:
filter_values = None  # TODO: por ejemplo [2, 4, 8, 12, 16, 24]
kernel_values = None  # TODO: por ejemplo [3, 5, 10, 15, 20]

# TODO: recorrer combinaciones, entrenar y evaluar.
# overfit_results = []
# for n_filters in filter_values:
#     for kernel_size in kernel_values:
#         ...

# TODO: convertir a DataFrame.
# overfit_df = pd.DataFrame(overfit_results)

# TODO: graficar performance y brecha train-test.

raise NotImplementedError("Completar barrido de complejidad de la CNN.")


### Análisis 6.1

                Respondan acá:
                - ¿Qué pueden decir del overfitting en relación con la complejidad de la red?
                - ¿Cuál es la mejor red para discriminar el set de entrenamiento?
                - Si además buscan minimizar overfitting, ¿cambia la elección?
                - ¿Qué arquitectura llevarían a la discusión en clase?

## Consigna — Objetivo 2. Drop-out y Batch Normalization

Objetivo 2. Drop-out y Batch Normalization
Dentro e lo que se incluye entre las técnicas de regularización podemos encontrar aquellas que
alteran la estructura de la red (como el dropout) y aquellas que afectan la LF. Afectar la LF
como veremos más adelante, permite no sólo disminuir el overfitting, sino también guiar el
entrenamiento  de  la  red  a  condiciones  y,  por  lo  tanto,  resultados  de  interés  para  el
desarrollador.
Siguiendo con, como en el primer ejercicio, las modificaciones en la arquitectura de la red,
vamos a introducir ahora dos técnicas clásicas: Batch Normalization  y Dropout

Batch Normalization es una técnica que normaliza los inputs de cada capa, para que tengan
media cercana a 0 y varianza  de  1, que que esto  acelera  el entrenamiento,  estabiliza  el
gradiente y ayuda a prevenir el overfitting. Generalmente se ubican después de las capas
lineales o convolucionales y antes de la función de activación lo que permite normalizar las
activaciones de la capa anterior antes de que pasen por la no linealidad.
En el armado de la red con pytorch el comando es: nn.BatchNorm1d(inputs)
Si tomamos nuestra red convolucional original:
Esta era definida cómo
self.conv1 = nn.Conv1d(in_channels=4, out_channels=c, kernel_size=k)
self.fc1 = nn.Linear(conv_output_size, 1) 
Y en el  def forward(self, x):
 x = F.relu(self.conv1(x))
 x = x.view(x.size(0), -1)  
O sea ReLu luego de la capa convolucional y la condensación de la salida a 1 neurona
Agregando “batch Normalization”
 self.conv1 = nn.Conv1d(in_channels=4, out_channels=c, kernel_size=k)
 self.bn1 = nn.BatchNorm1d(c)
 self.fc1 = nn.Linear(conv_output_size, 1)  
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)  
        x = F.relu(x)
        x = x.view(x.size(0), -1)  
Note que en este caso la dimensión del Batch Norm en 1d son los “c” canales/kernesl de la
CNN. En el caso de una red simple, se normalizará por batch de acuerdo al número de salidas
que tena la capa lineal. Btach normalizaton no posee hiperparámetros.
Dropout. Esta técnica desactiva de forma aleatoria un porcentaje de neuronas durante el
entrenamiento para evitar la dependencia excesiva entre ellas. El dropout se configura en
pytorch  como  una  capa  adicional  (en  este  caso despues  de  la  capa  convolucional  y/o el
batch_norm) con (donde p=x, es el % de neuronas que “tira”, y es un hiperparámetro)
 self.dropout = nn.Dropout(p=0.5)
En el forward queda:
x = self.conv1(x)
       x = self.bn1(x)

x = F.relu(x)
       x = self.dropout(x)
  x = x.view(x.size(0), -1)  
        
Nota importante : En PyTorch, cuando se usa Dropout, hay que de que el modelo esté en
modo  de  evaluación  al  evaluar  un  conjunto  de  validación  Esto  se  logra  llamando  al
método .eval() del modelo. Al hacerlo, Dropout desactiva la desactivación aleatoria de neuronas
y funciona en modo "full" (sin eliminar conexiones).
Ejemplo práctico:
# Entrenamiento
model.train()  # Activa Dropout y BatchNorm para el entrenamiento
# Evaluación
model.eval()  # Desactiva Dropout y pone BatchNorm en modo evaluación
with torch.no_grad():  # Desactiva el cálculo de gradientes para ahorrar memoria
    eval_output = model(eval_X)
Resumen de los comandos:
model.train(): Activa Dropout y BatchNorm en modo de entrenamiento.
model.eval(): Desactiva Dropout y pone BatchNorm en modo de evaluación (usa los promedios 
acumulados durante el entrenamiento).
torch.no_grad(): Desactiva el cálculo de gradientes, lo que reduce el uso de memoria y acelera 
la evaluación.
Ejercicio 6.2. Seleccione del ejercicio anterior 1 (o 2) arquitecturas donde haya observado un
overfitting.  Incluya  primero  batch  normalization  y  luego  dropout  (juegue  un  poco  con  el
hiperparámetro p). Analice el efecto de estas modificaciones en: a) la velocidad del aprendizaje,
b) el overfitting y c) la capacidad final del modelo. ¿Es capaz ahora de obtener un mejor modelo
que antes?

### TODO 6.2.a — CNN con Batch Normalization y Dropout

                Seleccionen 1 o 2 arquitecturas del ejercicio 6.1 donde haya overfitting.

                Implementen una versión regularizada con:
                - `nn.BatchNorm1d(c)` después de la convolución y antes de ReLU.
                - `nn.Dropout(p=...)` después de ReLU.

                Recuerden usar:
                - `model.train()` durante entrenamiento.
                - `model.eval()` durante evaluación.
                - `torch.no_grad()` al evaluar.

In [ ]:
class DNA_CNN_BN_Dropout(nn.Module):
    def __init__(self, nt_len, n_filters=8, kernel_size=5, dropout_p=0.5, use_batchnorm=True):
        super().__init__()
        # TODO: definir conv1.
        # TODO: definir BatchNorm opcional.
        # TODO: definir Dropout.
        # TODO: calcular conv_output_size y definir fc1.
        raise NotImplementedError

    def forward(self, x):
        # TODO: conv -> batchnorm opcional -> ReLU -> dropout -> flatten -> fc -> sigmoid.
        return x


### TODO 6.2.b — Experimentos con BatchNorm y Dropout

                Comparen:
                - CNN base,
                - CNN + BatchNorm,
                - CNN + Dropout,
                - CNN + BatchNorm + Dropout.

                Jueguen con `dropout_p` y miren:
                - velocidad de aprendizaje,
                - brecha train/test,
                - capacidad final del modelo.

In [ ]:
dropout_values = None  # TODO: por ejemplo [0.1, 0.25, 0.5, 0.7]
selected_architectures = None  # TODO: arquitecturas elegidas desde 6.1.

# TODO: entrenar modelos regularizados.
# regularization_results = []
# for arch in selected_architectures:
#     for dropout_p in dropout_values:
#         ...

# TODO: construir tabla y gráficos comparativos.
# regularization_df = pd.DataFrame(regularization_results)

raise NotImplementedError("Completar comparación con BatchNorm y Dropout.")


### Análisis 6.2

                Respondan acá:
                - ¿BatchNorm cambia la velocidad de aprendizaje?
                - ¿Dropout reduce el overfitting?
                - ¿Cuál fue el mejor valor de `p`?
                - ¿Lograron un mejor modelo que antes?

## Consigna — Objetivo 3) Regularizaciones que modifican la Loss-Function (LF).

Objetivo 3) Regularizaciones que modifican la Loss-Function (LF).
Ahora vamos a trabajar en regularizaciones (o estrategias) que se basan en modificar la LF, en
primer lugar dentro de las regularizaciones  típicas (L1, L2, Grad Penalty, Elastic Net etc.)
vamos a ir por lo simple y es utilizar L2 que se encuentra implementada implícitamente en
pytorch y sòlo requiere agregar un hiperparámetro al optimizador. L2 lo que hace es penalizar
el cuadrado de los “w” (o sea busca que los pesos sean pequeños) generando modelos más
“smooth”.
L2 se incorpora en nuestro modelo como un parámetro en el optimizador (el weight_decay)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.01)

Ejercicio 6.3 (opcional). Incluya en los alguno(s) de los modelos anteriores L2 Regularización
Analice el efecto de estas modificaciones en: a) la velocidad del aprendizaje, b) el overfitting y
c) la capacidad final del modelo. Nota: pruebe en que casos L2 tiene más efecto (con o sin
dropout), ¿por qué?

### TODO 6.3 opcional — L2 regularización con `weight_decay`

                L2 se incorpora en Adam como:

                `optim.Adam(model.parameters(), lr=0.001, weight_decay=0.01)`

                Prueben distintos valores de `weight_decay`, con y sin dropout.

In [ ]:
weight_decay_values = None  # TODO: por ejemplo [0.0, 1e-5, 1e-4, 1e-3, 1e-2]

# TODO: elegir arquitectura/s regularizadas o base.
# TODO: entrenar modelos con distintos weight_decay.
# l2_results = []
# for weight_decay in weight_decay_values:
#     ...

# TODO: armar tabla y gráficos.
# l2_df = pd.DataFrame(l2_results)

raise NotImplementedError("Completar análisis opcional de L2 regularización.")


### Análisis 6.3

                Respondan acá:
                - ¿L2 cambia la velocidad de aprendizaje?
                - ¿Reduce overfitting?
                - ¿Funciona distinto con y sin dropout?
                - ¿Por qué creen que pasa eso?

## Consigna — Objetivo 4) Análisis de cantidad y balance de los datos.

Objetivo 4) Análisis de cantidad y balance de los datos.
En muchos problemas biológicos, como el presente, existe un desbalance natural en los datos.
O sea son muchas menos las secuencias done el FT se une al ADN, que aquellas donde no.
Por otro lado, este desbalance, también impacta en el tipo de modelo que podemos querer
obtener, pudiendo buscar desarrollar un modelo “sensible”, aquel que detecte muchas (casi
todas) las regiones de unión a costa de algunos FN, o un modelo “preciso” aquel que detecte
“pocas” pero de alta confianza. Este desbalance se puede introducir de manera natural en el
modelo modificando la LF, asignando un peso “diferencial” a cada categoría. 
En pytorch podemos usar para un clasificación binaria (como la presente) la siguiente LF
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(10.0)) 
Donde se crea una LF de BCE que combina el valor predicho en ”logits” con su categorìa y 
asigna un peso de 10 a la clase positiva. Cuanto mayor este numero mayor peso se dará a las 
secuencias positivas.
Ejercicio 6.4 Construya un set desbalanceado donde la relación positivas / negativas sea 1
entre 1/10 - 1/100 (o más) y analicé el efecto en la performance del modelo al utilizar la LF
propuesta y variar el hiperparámetro pos_weight. ¿que puede decir de la capacidad del modelo
para detectar nuevos sitios de unión del FT?
Para finalizar nuestras redes para ADN combinaremos todo lo aprendido en un escenario mas
realista. Para ello tenga en cuenta que usted cuenta con el BED con la información de los picos
del ChipSeq de todos los cromosomas, y que puede obtener las secuencias de los otros
cromosomas en https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/
Ejercicio 6.5 Utilizando las secuencias positivas de uno (o màs) cromosomas (excluyendo el
22) construya y entrene (utilizando lo apreniddo en los TPs 5 y 6) un modelo que sea capaz de
escanear el Chr22 (o una sección el mismo) y determinar los picos de unión e JunD al mismo.
Construya  un  gráfico  de  “Señal”  vs Chrom:pos  y  superpongalo  con  el  gràfico  equivalente
obtenido del BED file.
¿cuàntas de las 561 regiones es capaz de reconocer con umbral de no más de 5-10% de FP
(verificados)?
¿Detecta alguna region no cubierta en el BED?
¿Que puede decir de la calidad del modelo final?

### TODO 6.4.a — Dataset desbalanceado

                Construyan un set donde la relación positivas/negativas sea entre:
                - `1/10`
                - `1/100`
                - o más desbalanceada si quieren probar.

                Documenten cuántas secuencias de cada clase quedan en train y test.

In [ ]:
def crear_dataset_desbalanceado(ratio_pos_neg=1/10, nt_len=100, random_state=42):
    """Construye tensores y dataframes para un escenario desbalanceado."""
    raise NotImplementedError


imbalance_ratios = None  # TODO: por ejemplo [1/10, 1/25, 1/50, 1/100]

# TODO: crear datasets desbalanceados.
# imbalanced_datasets = {}
# for ratio in imbalance_ratios:
#     imbalanced_datasets[ratio] = crear_dataset_desbalanceado(...)

raise NotImplementedError("Completar construcción de datasets desbalanceados.")


### TODO 6.4.b — `BCEWithLogitsLoss` y `pos_weight`

                Para usar `nn.BCEWithLogitsLoss(pos_weight=...)`, el modelo debe devolver **logits** y no aplicar `sigmoid` en el `forward`.

                Completen una CNN variante que devuelva logits y comparen distintos valores de `pos_weight`.

In [ ]:
class DNA_CNN_Logits(nn.Module):
    def __init__(self, nt_len, n_filters=8, kernel_size=5):
        super().__init__()
        # TODO: definir arquitectura igual a la CNN base, pero sin sigmoid final.
        raise NotImplementedError

    def forward(self, x):
        # TODO: devolver logits crudos.
        return x


def entrenar_modelo_logits(model, train, train_y, test, test_y, pos_weight=1.0, epochs=100, batch_size=10, lr=0.001):
    """Entrena usando BCEWithLogitsLoss y pos_weight."""
    raise NotImplementedError


pos_weight_values = None  # TODO: por ejemplo [1, 2, 5, 10, 25, 50, 100]

# TODO: entrenar/evaluar con distintos pos_weight y ratios de desbalance.
# imbalance_results = []

raise NotImplementedError("Completar análisis con pos_weight.")


### Análisis 6.4

                Respondan acá:
                - ¿Cómo cambia la sensibilidad/recall al aumentar `pos_weight`?
                - ¿Qué pasa con precisión y falsos positivos?
                - ¿Qué pueden decir de la capacidad del modelo para detectar nuevos sitios de unión?

### TODO 6.5 — Modelo final y escaneo de chr22

                Para el escenario final:
                - entrenen con secuencias positivas de uno o más cromosomas excluyendo chr22,
                - escaneen chr22 o una sección,
                - generen una señal CNN vs `Chrom:pos`,
                - superpongan contra señal/picos del BED.

                Este ejercicio puede requerir descargar FASTA de otros cromosomas desde UCSC.

In [ ]:
# TODO: definir rutas a datos externos si descargan otros cromosomas.
OTHER_CHROMS_FASTA_DIR = None
ALL_CHROMS_BED_PATH = None


def cargar_positivos_otros_cromosomas(bed_path, fasta_dir, excluir_chr="chr22"):
    raise NotImplementedError


def entrenar_modelo_final(datos_entrenamiento, config):
    raise NotImplementedError


def escanear_region_chr22(model, chr22_sequence, window_size=100, step=1):
    """Devuelve posiciones y scores del modelo sobre ventanas del chr22."""
    raise NotImplementedError


def comparar_senal_con_bed(scan_df, bed_df, threshold):
    """Compara picos predichos con regiones del BED."""
    raise NotImplementedError


# TODO: ejecutar pipeline final.
# final_model = ...
# scan_df = ...
# comparison_df = ...

raise NotImplementedError("Completar modelo final y escaneo de chr22.")


### Análisis 6.5

                Respondan acá:
                - ¿Cuántas de las 561 regiones de chr22 reconoce el modelo?
                - ¿Con qué umbral y con qué porcentaje de falsos positivos?
                - ¿Detecta regiones no cubiertas en el BED?
                - ¿Qué pueden decir de la calidad del modelo final?

## Checklist final del TP6

                - [ ] Dataset train/test reproducible, con 500 positivos train y 61 positivos test si siguen la sugerencia.
                - [ ] Curvas train/test loss para evidenciar overfitting.
                - [ ] Métricas train/test: LF, Accuracy, ROC-AUC, matriz de confusión, PR-AUC.
                - [ ] Barrido de filtros `2–24` y kernel `3–20 nt`.
                - [ ] Modelos con BatchNorm y Dropout.
                - [ ] Análisis opcional de L2 con `weight_decay`.
                - [ ] Dataset desbalanceado y análisis con `pos_weight`.
                - [ ] Modelo final para escanear chr22 o una región.
                - [ ] Figuras y tablas guardadas en `results/`.
                - [ ] Respuestas escritas para discutir en clase.